# Knowledge Graph desde cero · de PDFs a un cerebro

## Lo que vamos a hacer

1. Descargar 5 papers científicos de arxiv
2. Trocearlos
3. **El LLM extrae entidades y relaciones** de cada chunk · con structured output, esquema controlado
4. Montar el grafo en **NetworkX** (in-memory, sin DB)
5. **Persistir** como GraphML + pickle (portable, recargable)
6. Embeber los nodos en **FAISS** para búsqueda vectorial sobre el cerebro
7. **Visualizar el cerebro** interactivo con `pyvis` (HTML embebido en el notebook)
8. Hacer **queries combinadas** vector + grafo + LLM

## Stack

- `langchain-experimental` · `LLMGraphTransformer` (extrae nodos + aristas con structured output)
- `networkx>=3` · grafo en memoria
- `pyvis>=0.3` · visualización interactiva
- `faiss-cpu` · vector search sobre nodos
- `gpt-5-mini` · extracción del grafo y query final

**Nota de versiones (octubre 2026)**: `langchain-experimental` (donde vive `LLMGraphTransformer`) y `langchain-community` (`PyPDFLoader`) están en modo *sunset*: siguen funcionando con `langchain-core` 1.x, pero al importarlos verás un `DeprecationWarning`. No hay sustituto directo de `LLMGraphTransformer` fuera de Neo4j, así que lo mantenemos; en producción, fija versiones.

## Coste medido (octubre 2026)

5 papers × 3 páginas iniciales = 15 páginas → **45 chunks**. Cada chunk = 1 llamada a `gpt-5-mini`. Medido: ~61k tokens de entrada y ~197k de salida (135k de ellos de razonamiento) ≈ **$0.40** y **~55 segundos** con la extracción async. En secuencial, a ~52 s por chunk, serían unos 40 minutos.

## Setup

In [1]:
%pip install -q langchain langchain-core langchain-community langchain-openai langchain-experimental networkx pyvis faiss-cpu pypdf requests langchain-text-splitters

In [2]:
import os, getpass, requests
from pathlib import Path
import numpy as np

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

## 1. Descargar 5 papers de arxiv

Elegimos papers **relacionados** entre sí (Transformers · BERT · GPT-3 · RAG · Self-RAG) para que el grafo resultante esté **conectado**: comparten autores, datasets y métodos.

In [3]:
CORPUS_DIR = Path("./kg_corpus")
CORPUS_DIR.mkdir(exist_ok=True)

PAPERS = {
    "attention.pdf": "https://arxiv.org/pdf/1706.03762.pdf",   # Attention Is All You Need
    "bert.pdf":      "https://arxiv.org/pdf/1810.04805.pdf",   # BERT
    "gpt3.pdf":      "https://arxiv.org/pdf/2005.14165.pdf",   # GPT-3
    "rag.pdf":       "https://arxiv.org/pdf/2005.11401.pdf",   # Original RAG
    "selfrag.pdf":   "https://arxiv.org/pdf/2310.11511.pdf",   # Self-RAG
}

for name, url in PAPERS.items():
    dest = CORPUS_DIR / name
    if not dest.exists():
        print(f"  ↓ {name}")
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
        r.raise_for_status()
        dest.write_bytes(r.content)
    print(f"  ✓ {name}  ({dest.stat().st_size // 1024} KB)")

  ✓ attention.pdf  (2163 KB)
  ✓ bert.pdf  (756 KB)
  ✓ gpt3.pdf  (6609 KB)
  ✓ rag.pdf  (864 KB)
  ✓ selfrag.pdf  (1372 KB)


## 2. Cargar y truncar · primeras 3 páginas de cada paper

Para tener un demo rápido y barato, nos quedamos con las **primeras 3 páginas** de cada paper (suelen contener: título, autores, abstract, introducción · que es lo más rico en entidades).

Si tienes tiempo y créditos, sube esto a `[:10]` o quita el slice.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

PAGES_PER_PAPER = 3
all_docs = []
for name in PAPERS:
    pages = PyPDFLoader(str(CORPUS_DIR / name)).load()[:PAGES_PER_PAPER]
    for page in pages:
        page.metadata["source"] = name
    all_docs.extend(pages)

print(f"  ✓ {len(all_docs)} páginas cargadas (de {len(PAPERS)} papers × {PAGES_PER_PAPER} pgs)")

/tmp/ipykernel_49479/3837139598.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


  ✓ 15 páginas cargadas (de 5 papers × 3 pgs)


## 3. Chunking moderado · `RecursiveCharacterTextSplitter`

Chunks de ~1500 caracteres. Más grandes que un RAG normal porque queremos que cada chunk tenga **contexto suficiente** para que el LLM pueda extraer relaciones (ej. "Vaswani et al. propusieron el Transformer en 2017").

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "],
)

chunks = splitter.split_documents(all_docs)

print(f"✓ {len(chunks)} chunks (de {len(all_docs)} páginas)")

✓ 45 chunks (de 15 páginas)


## 4. Extracción del grafo con LLM · `LLMGraphTransformer`

Aquí está la magia. `LLMGraphTransformer` (langchain-experimental) usa **structured output** para forzar al LLM a devolver `(nodes, relationships)` con un esquema válido.

**Trucos clave**:

- `allowed_nodes` · restringimos el espacio de tipos · si no, el LLM inventa categorías incoherentes ("Thing", "Object", etc.)
- `allowed_relationships` · idem, controlamos las relaciones para que sean comparables entre chunks
- `node_properties=True` · permite al LLM enriquecer cada nodo con propiedades adicionales

Sin estas restricciones, el grafo final es **ruido**. Con ellas, es un cerebro coherente.

In [6]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")

transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[
        "Method",        # técnica algorítmica · Transformer, RAG, Self-Reflection
        "Architecture",  # arquitectura concreta · GPT-3, BERT-base
        "Person",        # autor · "Vaswani", "Devlin"
        "Institution",   # afiliación · Google, OpenAI, Stanford
        "Dataset",       # dataset usado · WikiText, GLUE, BookCorpus
        "Concept",       # idea/concepto · attention, masked language modeling
        "Paper",         # otra publicación citada
    ],
    allowed_relationships=[
        "AUTHORED_BY",   # Paper -> Person
        "AFFILIATED_WITH", # Person -> Institution
        "USES",          # Method/Arch -> Method/Concept/Dataset
        "PROPOSED_BY",   # Method -> Person/Paper
        "BUILDS_ON",     # Method -> Method (mejora/se basa en)
        "EVALUATED_ON",  # Method -> Dataset
        "CITES",         # Paper -> Paper
    ],
    node_properties=True,
)

/tmp/ipykernel_49479/3528766164.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


### Ejecutar la extracción

Esto es la parte cara: **una llamada a LLM por chunk**. `gpt-5-mini` es un modelo de razonamiento y cada extracción con structured output tarda decenas de segundos: en una ejecución secuencial con `convert_to_graph_documents` medimos **361 s para 7 chunks (~52 s por chunk)**, así que los 5 papers en secuencial no caben en una clase.

💡 Por eso usamos la versión async, `aconvert_to_graph_documents`, que lanza todas las llamadas en paralelo: el tiempo total pasa a ser el de las llamadas más lentas, no la suma.

In [7]:
import time

t0 = time.time()
# versión async: lanza las llamadas al LLM en paralelo (en Jupyter se puede usar await directamente)
graph_documents = await transformer.aconvert_to_graph_documents(chunks)
elapsed = time.time() - t0

n_nodes = sum(len(gd.nodes) for gd in graph_documents)
n_rels = sum(len(gd.relationships) for gd in graph_documents)
print(f"  ✓ extracción terminada en {elapsed:.0f}s")
print(f"  · {n_nodes} nodos extraídos · {n_rels} relaciones")
print(f"  · ~{elapsed/len(chunks):.1f}s por chunk")

  ✓ extracción terminada en 56s
  · 788 nodos extraídos · 799 relaciones
  · ~1.2s por chunk


### Echar un vistazo a un chunk del grafo

In [8]:
# Inspeccionamos el primer chunk: nodos y relaciones que el LLM extrajo de él
first = graph_documents[0]
print(f"Source: {first.source.metadata.get('source')}\n")

print("Nodos:")
for n in first.nodes[:8]:
    print(f"  - [{n.type}] {n.id}")

print("\nRelaciones:")
for r in first.relationships[:8]:
    print(f"  - ({r.source.id}) -[{r.type}]-> ({r.target.id})")

Source: attention.pdf

Nodos:
  - [Paper] Attention Is All You Need
  - [Architecture] Transformer
  - [Concept] Attention Mechanism
  - [Dataset] Wmt 2014 English-To-German
  - [Dataset] Wmt 2014 English-To-French
  - [Person] Ashish Vaswani
  - [Person] Noam Shazeer
  - [Person] Niki Parmar

Relaciones:
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Ashish Vaswani)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Noam Shazeer)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Niki Parmar)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Jakob Uszkoreit)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Llion Jones)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Aidan N. Gomez)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Łukasz Kaiser)
  - (Attention Is All You Need) -[AUTHORED_BY]-> (Illia Polosukhin)


## 5. Construir el grafo en NetworkX

Pasamos de la representación de Langchain (lista de `GraphDocument`) a un `MultiDiGraph` de NetworkX. NetworkX nos da:

- Algoritmos clásicos de grafos (shortest path, centralidad, comunidades)
- Persistencia estándar (GraphML, pickle, JSON)
- Integración con `pyvis` para visualización

**Importante**: `MultiDiGraph` permite múltiples aristas distintas entre los mismos nodos (útil porque el LLM puede extraer varias relaciones diferentes entre dos entidades).

In [9]:
import networkx as nx

G = nx.MultiDiGraph()

for gd in graph_documents:
    for node in gd.nodes:
        # Si el nodo ya existe, conservamos type del primer encuentro
        if not G.has_node(node.id):
            G.add_node(node.id, type=node.type)
    for rel in gd.relationships:
        G.add_edge(rel.source.id, rel.target.id, type=rel.type)

print(f"  · nodos únicos     : {G.number_of_nodes()}")
print(f"  · aristas totales  : {G.number_of_edges()}")
print(f"  · conexión media   : {2 * G.number_of_edges() / G.number_of_nodes():.1f} aristas/nodo")

  · nodos únicos     : 666
  · aristas totales  : 799
  · conexión media   : 2.4 aristas/nodo


### Top 10 nodos más conectados · ¿quién manda en este cerebro?

In [10]:
degree = dict(G.degree())
top = sorted(degree.items(), key=lambda x: -x[1])[:10]

print("Top 10 nodos por conectividad:\n")
for nid, deg in top:
    ntype = G.nodes[nid].get("type", "?")
    print(f"  [{ntype:14s}] {nid:30s} · {deg} conexiones")

Top 10 nodos por conectividad:

  [Method        ] Self-Rag                       · 49 conexiones
  [Paper         ] Language Models Are Few-Shot Learners · 36 conexiones
  [Method        ] Bert                           · 34 conexiones
  [Institution   ] Openai                         · 32 conexiones
  [Method        ] Retrieval-Augmented Generation (Rag) · 27 conexiones
  [Architecture  ] Transformer                    · 22 conexiones
  [Paper         ] Retrieval-Augmented Generation For Knowledge-Intensive Nlp Tasks · 20 conexiones
  [Concept       ] Reflection Tokens              · 18 conexiones
  [Paper         ] Preprint                       · 18 conexiones
  [Paper         ] Contents                       · 17 conexiones


### Distribución por tipo de nodo

In [11]:
from collections import Counter

type_counts = Counter(G.nodes[n].get("type", "?") for n in G.nodes())
print("Distribución por tipo:\n")
for t, c in type_counts.most_common():
    print(f"  {t:14s} : {c}")

Distribución por tipo:

  Concept        : 298
  Paper          : 115
  Person         : 94
  Architecture   : 56
  Method         : 48
  Dataset        : 41
  Institution    : 13
  ?              : 1


## 6. Persistir el grafo · GraphML + pickle

Dos formatos:

- **GraphML** · XML estándar · portable, abrible con Gephi/yEd/Cytoscape
- **pickle** · Python nativo · recarga inmediata, preserva tipos exactos

Para producción usarías Neo4j (como en el notebook 06). Para investigación / análisis exploratorio, GraphML es perfecto.

In [12]:
import pickle

OUT_DIR = Path("./kg_artifacts")
OUT_DIR.mkdir(exist_ok=True)

# 1. GraphML
nx.write_graphml(G, OUT_DIR / "brain.graphml")

# 2. pickle
with open(OUT_DIR / "brain.pkl", "wb") as f:
    pickle.dump(G, f)

print(f"  ✓ guardado en {OUT_DIR}/")
print(f"    - brain.graphml  ({(OUT_DIR / 'brain.graphml').stat().st_size // 1024} KB)")
print(f"    - brain.pkl      ({(OUT_DIR / 'brain.pkl').stat().st_size // 1024} KB)")

  ✓ guardado en kg_artifacts/
    - brain.graphml  (152 KB)
    - brain.pkl      (88 KB)


## 7. Embeddings de nodos · búsqueda vectorial sobre el cerebro

Indexamos cada nodo con su embedding en FAISS. Esto nos da **búsqueda semántica de entidades** · puedes preguntar "¿qué nodos hablan de modelos generativos?" y FAISS te devuelve los nodos más cercanos al concepto, aunque no se llamen así literalmente.

Es el bridge entre el mundo **simbólico** (grafo) y el **vectorial** (embeddings).

In [13]:
from langchain_openai import OpenAIEmbeddings
import faiss

embedder = OpenAIEmbeddings(model="text-embedding-3-small")

node_ids = list(G.nodes())
# Texto enriquecido por nodo: "id (type)" facilita que el embedder capture contexto
node_texts = [
    f"{nid} ({G.nodes[nid].get('type', 'unknown')})"
    for nid in node_ids
]

node_vecs = np.array(embedder.embed_documents(node_texts), dtype=np.float32)
faiss.normalize_L2(node_vecs)  # para usar producto escalar = coseno

node_index = faiss.IndexFlatIP(node_vecs.shape[1])
node_index.add(node_vecs)

# Persistimos también el índice
faiss.write_index(node_index, str(OUT_DIR / "node_index.faiss"))
with open(OUT_DIR / "node_ids.pkl", "wb") as f:
    pickle.dump(node_ids, f)

print(f"  ✓ {len(node_ids)} nodos indexados en FAISS")

  ✓ 666 nodos indexados en FAISS


In [14]:
def search_nodes(query: str, k: int = 5):
    """Búsqueda vectorial sobre los nodos del cerebro."""
    q = np.array([embedder.embed_query(query)], dtype=np.float32)
    faiss.normalize_L2(q)
    scores, idx = node_index.search(q, k)
    return [(node_ids[i], float(scores[0][j])) for j, i in enumerate(idx[0])]

print("Búsqueda: 'modelos generativos pre-entrenados'\n")
for nid, s in search_nodes("modelos generativos pre-entrenados", k=5):
    ntype = G.nodes[nid].get("type", "?")
    print(f"  [{ntype:14s}] {nid:35s}  score {s:.3f}")

print("\nBúsqueda: 'mecanismos de atención'\n")
for nid, s in search_nodes("mecanismos de atención", k=5):
    ntype = G.nodes[nid].get("type", "?")
    print(f"  [{ntype:14s}] {nid:35s}  score {s:.3f}")

Búsqueda: 'modelos generativos pre-entrenados'

  [Architecture  ] Generative Pre-Trained Transformer (Openai Gpt)  score 0.512
  [Concept       ] Text Generation Models               score 0.493
  [Concept       ] Pre-Trained_Neural_Language_Models   score 0.483
  [Concept       ] Pre-Trained Language Models          score 0.482
  [?             ] Pre-Trained Models With A Differentiable Access Mechanism To Explicit Non-Parametric Memory  score 0.481

Búsqueda: 'mecanismos de atención'



  [Method        ] Attention Mechanisms                 score 0.481
  [Concept       ] Attention Mechanism                  score 0.454
  [Concept       ] Recurrent Attention Mechanism        score 0.444
  [Concept       ] Multi-Head Attention                 score 0.359
  [Concept       ] Pre-Trained Access Mechanisms        score 0.357


## 8. Visualizar el cerebro · `pyvis` interactivo

Aquí está el momento WOW. `pyvis` toma el grafo de NetworkX y lo renderiza como HTML interactivo: nodos arrastrables, zoom, tooltips, físicas estilo *force-directed*.

Coloreamos los nodos por tipo para que el cerebro sea legible de un vistazo.

In [15]:
from pyvis.network import Network

TYPE_COLORS = {
    "Method":       "#3498db",  # azul
    "Architecture": "#9b59b6",  # morado
    "Person":       "#e74c3c",  # rojo
    "Institution":  "#34495e",  # gris oscuro
    "Dataset":      "#2ecc71",  # verde
    "Concept":      "#f39c12",  # naranja
    "Paper":        "#1abc9c",  # turquesa
}
DEFAULT_COLOR = "#95a5a6"

# Anotamos cada nodo con color y tooltip
for nid in G.nodes():
    ntype = G.nodes[nid].get("type", "?")
    G.nodes[nid]["color"] = TYPE_COLORS.get(ntype, DEFAULT_COLOR)
    G.nodes[nid]["title"] = f"{nid}\nTipo: {ntype}\nGrado: {G.degree(nid)}"
    G.nodes[nid]["size"] = 10 + min(G.degree(nid), 30)  # tamaño proporcional a grado

# Anotamos las aristas con label
for u, v, key, data in G.edges(keys=True, data=True):
    data["label"] = data.get("type", "")
    data["title"] = data.get("type", "")

net = Network(
    height="700px",
    width="100%",
    directed=True,
    notebook=True,
    cdn_resources="in_line",
    bgcolor="#ffffff",
    font_color="#222222",
)
net.from_nx(G)
net.barnes_hut(spring_length=180, spring_strength=0.005)  # físicas suaves

html_path = str(OUT_DIR / "brain.html")
net.write_html(html_path, notebook=False)
print(f"  ✓ visualización guardada en {html_path}")

  ✓ visualización guardada en kg_artifacts/brain.html


### Renderizar inline · arrastra los nodos para explorar

In [16]:
from IPython.display import IFrame

# Mostramos el grafo embebido (puedes hacer pan, zoom, click en nodos)
IFrame(src=html_path, width="100%", height="720")

## 9. Queries combinadas · vector + grafo

Lo poderoso del setup: combinar búsqueda vectorial con traversal del grafo.

**Patrón**: "dame todo lo que el cerebro sabe sobre X" =
1. Buscar el nodo más cercano semánticamente a X (FAISS)
2. Extraer el subgrafo de ese nodo y sus vecinos (NetworkX)
3. Pasar ese subgrafo como contexto al LLM

In [17]:
def neighborhood(node_id: str, depth: int = 1):
    """Devuelve el subgrafo de un nodo + sus vecinos a distancia `depth`."""
    if not G.has_node(node_id):
        return None
    nodes = {node_id}
    frontier = {node_id}
    for _ in range(depth):
        new_frontier = set()
        for n in frontier:
            new_frontier.update(G.predecessors(n))
            new_frontier.update(G.successors(n))
        frontier = new_frontier - nodes
        nodes |= frontier
    return G.subgraph(nodes)

# Demo
results = search_nodes("transformer", k=1)
best_node = results[0][0]
sub = neighborhood(best_node, depth=1)

print(f"Nodo más cercano a 'transformer': {best_node}")
print(f"Subgrafo (vecinos a depth=1): {sub.number_of_nodes()} nodos, {sub.number_of_edges()} aristas")
print("\nRelaciones del subgrafo:")
for u, v, data in sub.edges(data=True):
    print(f"  ({u}) -[{data.get('type', '?')}]-> ({v})")

Nodo más cercano a 'transformer': Transformers
Subgrafo (vecinos a depth=1): 2 nodos, 1 aristas

Relaciones del subgrafo:
  (Bert) -[BUILDS_ON]-> (Transformers)


### Pregunta al cerebro · LLM + subgrafo como contexto

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ask_prompt = ChatPromptTemplate.from_template(
    "Eres un experto en NLP. Te paso un fragmento de un grafo de conocimiento extraído "
    "de papers científicos. Usa SOLO esa información para responder.\n\n"
    "Subgrafo:\n{subgraph}\n\n"
    "Pregunta: {question}"
)
ask_chain = ask_prompt | ChatOpenAI(model="gpt-5-mini") | StrOutputParser()

def ask_brain(question: str, k_nodes: int = 3, depth: int = 1):
    # 1. Encontrar nodos más relevantes vía vector search
    seeds = [nid for nid, _ in search_nodes(question, k=k_nodes)]
    # 2. Extraer subgrafo unión de sus vecindarios
    nodes = set(seeds)
    for s in seeds:
        sub = neighborhood(s, depth=depth)
        if sub is not None:
            nodes |= set(sub.nodes())
    sub = G.subgraph(nodes)
    # 3. Serializar subgrafo en texto
    triples = [
        f"({u}) -[{d.get('type', '?')}]-> ({v})"
        for u, v, d in sub.edges(data=True)
    ]
    subgraph_text = "\n".join(triples) if triples else "(grafo vacío)"
    return ask_chain.invoke({"subgraph": subgraph_text, "question": question})

print(ask_brain("¿Qué papers usan attention?"))
print("\n---\n")
print(ask_brain("¿Qué arquitecturas se construyen sobre Transformers?"))

Según el subgrafo, el Transformer usa la Attention Mechanism (Transformer -[USES]-> Attention Mechanism). El Transformer fue propuesto por el paper "Attention Is All You Need" (Transformer -[PROPOSED_BY]-> Attention Is All You Need). Por tanto, el paper "Attention Is All You Need" es el que usa attention en este subgrafo.

---



Según el subgrafo, las arquitecturas que se construyen sobre Transformers son:

- Openai Gpt
- Bert


## Cierre

Lo que has construido:

- Un **pipeline completo** de PDFs → Knowledge Graph sin tocar Neo4j
- Un cerebro **persistente** (GraphML + pickle) que puedes recargar mañana
- Un cerebro **buscable** (FAISS sobre nodos) que mezcla vectorial + simbólico
- Un cerebro **visible** (pyvis interactivo) que puedes enseñar a otros
- Un cerebro **interrogable** (LLM + subgrafo) que responde preguntas naturales

**Cuándo usar este patrón en lugar de Neo4j**:

| Caso | Recomendación |
|---|---|
| Investigación / análisis ad hoc | **NetworkX (este notebook)** |
| Grafo < 100k nodos | **NetworkX en RAM funciona perfecto** |
| Producción multi-usuario | **Neo4j** (otro notebook) |
| Necesitas Cypher / GraphQL | **Neo4j** |
| Quieres un demo standalone que vivir en un notebook | **NetworkX** |

**Mejoras que puedes hacer**:

- Procesar más páginas (`PAGES_PER_PAPER = 10` o `None` para todo)
- Refinar el esquema (`allowed_nodes` / `allowed_relationships`) para tu dominio
- Detectar y fusionar entidades duplicadas (ej. "GPT-3" y "GPT3") con embedding clustering
- Calcular **comunidades** con `nx.community` para encontrar clusters temáticos
- Subir todo a Neo4j Aura cuando crezca con `Neo4jGraph.add_graph_documents()`